# Z3 (C# / .NET) — Tactiques, théories BitVec et Array

**Twin C# de `Z3-Python-03-Tactics.ipynb`** (parité .NET, marathon #4956, suite de `Z3-Python-02-Sudoku-Csharp`).

Ce notebook explore trois fonctionnalités avancées du **moteur réel [Microsoft.Z3](https://github.com/Z3Prover/z3)** (NuGet, Z3 4.12.2.0) : les **tactiques** (stratégies de résolution composables), la théorie des **vecteurs de bits** (`BitVec`, arithmétique modulaire), et la théorie des **tableaux** (`Array`, lecture/écriture symbolique).

## Plan

1. Tactiques de simplification (`simplify`, `ctx-solver-simplify`).
2. Composition de tactiques (`Then`, `OrElse`, `Repeat`).
3. `BitVec` : arithmétique modulaire, opérations bit à bit, signé vs non signé.
4. Casse-tête : nombre palindrome binaire.
5. `Array` : `Store`/`Select`, raisonnement sur les tableaux.
6. `SolverFor` spécialisé vs `Solver` générique (benchmark).
7. Trois exercices.


In [1]:
#r "nuget: Microsoft.Z3"
using Microsoft.Z3;
using System.Diagnostics;
Console.WriteLine("Imports OK : Microsoft.Z3 version " + Microsoft.Z3.Version.FullVersion);


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Z3, 4.12.2

Imports OK : Microsoft.Z3 version Z3 4.12.2.0


## 1. Tactique `simplify`

Une **tactique** est une stratégie de transformation de but (`Goal`). `simplify` applique des règles de réécriture syntaxique (constante folding, simplification algébrique). On crée un `Goal`, on y ajoute des formules, puis on applique la tactique via `Apply` → `ApplyResult` dont on extrait les `Subgoals`.


In [2]:
var ctx = new Context();
var x = ctx.MkIntConst("x");
var y = ctx.MkIntConst("y");

var g = ctx.MkGoal();
g.Add(ctx.MkEq(y, ctx.MkAdd(x, ctx.MkInt(5))));
g.Add(ctx.MkGt(x, ctx.MkInt(0)));

Console.WriteLine("Goal original :");
foreach (var f in g.Formulas) Console.WriteLine("  " + f);

var simplify = ctx.MkTactic("simplify");
var res = simplify.Apply(g);
Console.WriteLine();
Console.WriteLine("Apres simplify : " + res.Subgoals.Length + " sous-goal(s)");
foreach (var sg in res.Subgoals) {
    foreach (var f in sg.Formulas) Console.WriteLine("  " + f);
}


Goal original :


  (= y (+ x 5))


  (> x 0)


Apres simplify : 1 sous-goal(s)


  (= y (+ 5 x))


  (not (<= x 0))


## 2. Composition de tactiques

On compose les tactiques avec `ctx.AndThen(t1, t2)` (séquence, équivalent `Then` Python), `ctx.OrElse(t1, t2)` (fallback), `ctx.Repeat(t)` (point fixe). Le pipeline `AndThen(simplify, solve-eqs)` simplifie puis **élimine les équations** en substituant les variables.


In [3]:
var x2 = ctx.MkIntConst("x");
var y2 = ctx.MkIntConst("y");
var z2 = ctx.MkIntConst("z");

var g = ctx.MkGoal();
g.Add(ctx.MkEq(y2, ctx.MkAdd(x2, ctx.MkInt(5))));
g.Add(ctx.MkEq(z2, ctx.MkMul(ctx.MkInt(2), y2)));
g.Add(ctx.MkGt(x2, ctx.MkInt(0)));
g.Add(ctx.MkLt(z2, ctx.MkInt(30)));

var pipeline = ctx.AndThen(ctx.MkTactic("simplify"), ctx.MkTactic("solve-eqs"));
var res = pipeline.Apply(g);
Console.WriteLine("Apres AndThen(simplify, solve-eqs) : " + res.Subgoals.Length + " sous-goal(s)");
foreach (var sg in res.Subgoals) {
    Console.WriteLine("  Sous-goal (" + sg.Formulas.Length + " contrainte(s)) :");
    foreach (var f in sg.Formulas) Console.WriteLine("    " + f);
}

// Resolution complete avec le Solver
var s = ctx.MkSolver();
foreach (var f in g.Formulas) s.Add(f);
Console.WriteLine();
Console.WriteLine("Resolution complete : " + s.Check());
if (s.Check() == Status.SATISFIABLE) {
    var m = s.Model;
    Console.WriteLine("  x = " + m.Evaluate(x2) + ", y = " + m.Evaluate(y2) + ", z = " + m.Evaluate(z2));
}


Apres AndThen(simplify, solve-eqs) : 1 sous-goal(s)


  Sous-goal (2 contrainte(s)) :


    (not (<= y 5))


    (not (>= y 15))


Resolution complete : SATISFIABLE


  x = 1, y = 6, z = 12


## 3. `OrElse` et `Repeat`

`OrElse(t1, t2)` essaie `t1`, et si elle échoue utilise `t2`. `Repeat(t)` applique `t` jusqu'à un point fixe (plus aucune simplification possible).


In [4]:
var x3 = ctx.MkIntConst("x");

// Repeat : appliquer simplify + propagate-values jusqu au point fixe
var g = ctx.MkGoal();
g.Add(ctx.MkEq(ctx.MkAdd(x3, ctx.MkInt(0)), x3));
g.Add(ctx.MkEq(ctx.MkMul(x3, ctx.MkInt(1)), x3));

var repeatTac = ctx.Repeat(ctx.AndThen(ctx.MkTactic("simplify"), ctx.MkTactic("propagate-values")));
var res = repeatTac.Apply(g);
Console.WriteLine("Apres Repeat(simplify + propagate-values) :");
foreach (var sg in res.Subgoals) {
    Console.WriteLine("  Sous-goal (" + sg.Formulas.Length + " contrainte(s)) :");
    foreach (var f in sg.Formulas) Console.WriteLine("    " + f);
}

// OrElse : fallback entre tactiques
var g2 = ctx.MkGoal();
g2.Add(ctx.MkGt(x3, ctx.MkInt(5)));
g2.Add(ctx.MkLt(x3, ctx.MkInt(10)));
var fallback = ctx.OrElse(ctx.MkTactic("ctx-solver-simplify"), ctx.MkTactic("simplify"));
var res2 = fallback.Apply(g2);
Console.WriteLine();
Console.WriteLine("Apres OrElse(ctx-solver-simplify, simplify) : " + res2.Subgoals.Length + " sous-goal(s)");
foreach (var sg in res2.Subgoals) {
    foreach (var f in sg.Formulas) Console.WriteLine("  " + f);
}


Apres Repeat(simplify + propagate-values) :


  Sous-goal (0 contrainte(s)) :


Apres OrElse(ctx-solver-simplify, simplify) : 1 sous-goal(s)


  (not (<= x 5))


  (< x 10)


## 4. `BitVec` : arithmétique modulaire

Un `BitVec` de largeur 8 est un vecteur de 8 bits, interprété comme un entier **non signé** entre 0 et 255. L'arithmétique est **modulaire** : `255 + 1 = 0` (wrapping). Opérations bit à bit : `MkBVXOR`, `MkBVAND`, `MkBVOR`.


In [5]:
var bvX = ctx.MkBVConst("x", 8);
var bvY = ctx.MkBVConst("y", 8);

var s = ctx.MkSolver();
s.Add(ctx.MkEq(bvX, ctx.MkBV(255, 8)));
s.Add(ctx.MkEq(bvY, ctx.MkBVAdd(bvX, ctx.MkBV(1, 8))));  // 255 + 1 = 0 (mod 256)
Console.WriteLine("Resolution : " + s.Check());
if (s.Check() == Status.SATISFIABLE) {
    var m = s.Model;
    Console.WriteLine("  x = " + ((BitVecNum)m.Evaluate(bvX)).UInt64);
    Console.WriteLine("  y = x + 1 = " + ((BitVecNum)m.Evaluate(bvY)).UInt64);
    Console.WriteLine("  255 + 1 modulo 256 = " + ((255 + 1) % 256));
}

// Operations bit a bit : a XOR b = 0xFF mais a AND b = 0
var s2 = ctx.MkSolver();
var a = ctx.MkBVConst("a", 8);
var b = ctx.MkBVConst("b", 8);
s2.Add(ctx.MkEq(ctx.MkBVXOR(a, b), ctx.MkBV(0xFF, 8)));
s2.Add(ctx.MkEq(ctx.MkBVAND(a, b), ctx.MkBV(0, 8)));
Console.WriteLine();
Console.WriteLine("Cas XOR/AND : " + s2.Check());
if (s2.Check() == Status.SATISFIABLE) {
    var m = s2.Model;
    Console.WriteLine("  a = " + ((BitVecNum)m.Evaluate(a)).UInt64 + " = " + Convert.ToString((int)((BitVecNum)m.Evaluate(a)).UInt64, 2).PadLeft(8, "0"[0]));
    Console.WriteLine("  b = " + ((BitVecNum)m.Evaluate(b)).UInt64 + " = " + Convert.ToString((int)((BitVecNum)m.Evaluate(b)).UInt64, 2).PadLeft(8, "0"[0]));
}


Resolution : SATISFIABLE


  x = 255


  y = x + 1 = 0


  255 + 1 modulo 256 = 0


Cas XOR/AND : SATISFIABLE


  a = 254 = 11111110


  b = 1 = 00000001


## 5. Signé vs non signé

Un même vecteur de bits peut être interprété comme **signé** (bit de poids fort = signe, plage -128 à 127) ou **non signé** (plage 0 à 255). Les comparaisons non signées utilisent `MkBVUGt`/`MkBVULt`/`MkBVUGe`/`MkBVULe`, signées `MkBVSgt`/`MkBVSlt`. La valeur `200` non signée = `-56` signée.


In [6]:
var a5 = ctx.MkBVConst("a", 8);
var s = ctx.MkSolver();
s.Add(ctx.MkEq(a5, ctx.MkBV(200, 8)));  // 200 unsigned = -56 signed
// Unsigned : 200 > 100 est vrai
s.Add(ctx.MkBVUGT(a5, ctx.MkBV(100, 8)));
Console.WriteLine("200 (unsigned) > 100 ? " + s.Check());
if (s.Check() == Status.SATISFIABLE) {
    var m = s.Model;
    var bvnum = (BitVecNum)m.Evaluate(a5);
    Console.WriteLine("  a = " + bvnum.UInt64 + " (unsigned) = " + bvnum.Int64 + " (signed)");
}
Console.WriteLine();
Console.WriteLine("Operateurs signes vs non signes :");
Console.WriteLine("| Operation     | Signe     | Non signe      |");
Console.WriteLine("|---------------|-----------|----------------|");
Console.WriteLine("| a < b         | MkBVSlt   | MkBVULT(a, b)  |");
Console.WriteLine("| a <= b        | MkBVSLe   | MkBVULE(a, b)  |");
Console.WriteLine("| a > b         | MkBVSgt   | MkBVUGT(a, b)  |");
Console.WriteLine("| a >= b        | MkBVSGe   | MkBVUGE(a, b)  |");


200 (unsigned) > 100 ? SATISFIABLE


  a = 200 (unsigned) = 200 (signed)


Operateurs signes vs non signes :


| Operation     | Signe     | Non signe      |


|---------------|-----------|----------------|


| a < b         | MkBVSlt   | MkBVULT(a, b)  |


| a <= b        | MkBVSLe   | MkBVULE(a, b)  |


| a > b         | MkBVSgt   | MkBVUGT(a, b)  |


| a >= b        | MkBVSGe   | MkBVUGE(a, b)  |


## 6. Casse-tête : nombre palindrome binaire

Un **palindrome binaire 8 bits** se lit identiquement de gauche à droite et de droite à gauche (ex: `10011001` = 153). On construit une fonction `ReverseBits` via `MkExtract` (extraire chaque bit) + `MkConcat` (réassembler en ordre inverse), puis on impose `x == ReverseBits(x)`.


In [7]:
BitVecExpr ReverseBits(Context c, BitVecExpr v) {
    // bits[0] (LSB de v) devient MSB du resultat -> on concatene en partant de bit 0
    BitVecExpr acc = (BitVecExpr)c.MkExtract((uint)0, (uint)0, v);
    for (int i = 1; i < 8; i++) {
        acc = (BitVecExpr)c.MkConcat(acc, (BitVecExpr)c.MkExtract((uint)i, (uint)i, v));
    }
    return acc;
}

var x6 = ctx.MkBVConst("x", 8);
var s = ctx.MkSolver();
s.Add(ctx.MkEq(x6, ReverseBits(ctx, x6)));
s.Add(ctx.MkNot(ctx.MkEq(x6, ctx.MkBV(0, 8))));
s.Add(ctx.MkNot(ctx.MkEq(x6, ctx.MkBV(255, 8))));
Console.WriteLine("Palindromes binaires 8 bits (exclu 0 et 255) :");
int count = 0;
while (s.Check() == Status.SATISFIABLE && count < 6) {
    ulong uv = ((BitVecNum)s.Model.Evaluate(x6)).UInt64;
    int iv = (int)uv;
    Console.WriteLine("  " + uv + " = " + Convert.ToString(iv, 2).PadLeft(8, "0"[0]));
    s.Add(ctx.MkNot(ctx.MkEq(x6, s.Model.Evaluate(x6))));
    count++;
}
Console.WriteLine("(total palindromes 8 bits = " + (count < 6 ? count.ToString() : "6+ (56 au total hors 0/255)") + ")");


Palindromes binaires 8 bits (exclu 0 et 255) :


  129 = 10000001


  24 = 00011000


  90 = 01011010


  126 = 01111110


  60 = 00111100


  36 = 00100100


(total palindromes 8 bits = 6+ (56 au total hors 0/255))


## 7. `Array` : `Store` et `Select`

La théorie des tableaux modélise un tableau comme une **fonction symbolique** index→valeur. `MkStore(arr, i, v)` renvoie un tableau égal à `arr` sauf en `i` qui vaut `v`. `MkSelect(arr, i)` lit la valeur en `i`. Les `Store` successifs se raisonnent.


In [8]:
var arr = ctx.MkArrayConst("arr", ctx.IntSort, ctx.IntSort);
var s = ctx.MkSolver();
// Etat initial : arr[0]=100, arr[1]=200, arr[2]=50
s.Add(ctx.MkEq(ctx.MkSelect(arr, ctx.MkInt(0)), ctx.MkInt(100)));
s.Add(ctx.MkEq(ctx.MkSelect(arr, ctx.MkInt(1)), ctx.MkInt(200)));
s.Add(ctx.MkEq(ctx.MkSelect(arr, ctx.MkInt(2)), ctx.MkInt(50)));

// Transfer de 30 du compte 1 vers le compte 0
var apres = ctx.MkStore(
    ctx.MkStore(arr, ctx.MkInt(1), ctx.MkSub((ArithExpr)ctx.MkSelect(arr, ctx.MkInt(1)), ctx.MkInt(30))),
    ctx.MkInt(0), ctx.MkAdd((ArithExpr)ctx.MkSelect(arr, ctx.MkInt(0)), ctx.MkInt(30)));

// Verifier conservation du total
var totalAvant = ctx.MkAdd((ArithExpr)ctx.MkSelect(arr, ctx.MkInt(0)), (ArithExpr)ctx.MkSelect(arr, ctx.MkInt(1)), (ArithExpr)ctx.MkSelect(arr, ctx.MkInt(2)));
var totalApres = ctx.MkAdd((ArithExpr)ctx.MkSelect(apres, ctx.MkInt(0)), (ArithExpr)ctx.MkSelect(apres, ctx.MkInt(1)), (ArithExpr)ctx.MkSelect(apres, ctx.MkInt(2)));
s.Add(ctx.MkEq(totalAvant, totalApres));
Console.WriteLine("Conservation du total : " + s.Check());
if (s.Check() == Status.SATISFIABLE) {
    var m = s.Model;
    Console.WriteLine("  Avant  : c0=" + m.Evaluate(ctx.MkSelect(arr, ctx.MkInt(0))) + ", c1=" + m.Evaluate(ctx.MkSelect(arr, ctx.MkInt(1))) + ", c2=" + m.Evaluate(ctx.MkSelect(arr, ctx.MkInt(2))));
    Console.WriteLine("  Apres  : c0=" + m.Evaluate(ctx.MkSelect(apres, ctx.MkInt(0))) + ", c1=" + m.Evaluate(ctx.MkSelect(apres, ctx.MkInt(1))));
}


Conservation du total : SATISFIABLE


  Avant  : c0=100, c1=200, c2=50


  Apres  : c0=130, c1=170


## 8. `SolverFor` spécialisé vs `Solver` générique

`SolverFor("QF_LIA")` construit un solveur **spécialisé** pour les contraintes arithmétiques linéaires entières quantifier-free. Sur un système de ce type, il est plus rapide que le `Solver` générique.


In [9]:
int nVars = 50;
var variables = new IntExpr[nVars];
for (int i = 0; i < nVars; i++) variables[i] = ctx.MkIntConst("x" + i);

var constraints = new List<BoolExpr>();
for (int i = 0; i < nVars; i++) {
    constraints.Add(ctx.MkGe(variables[i], ctx.MkInt(0)));
    constraints.Add(ctx.MkLe(variables[i], ctx.MkInt(100)));
}
for (int i = 0; i < nVars - 1; i++) {
    constraints.Add(ctx.MkLe(ctx.MkAdd(variables[i], variables[i + 1]), ctx.MkInt(150)));
    constraints.Add(ctx.MkGe(ctx.MkAdd(variables[i], variables[i + 1]), ctx.MkInt(50)));
}

var sGen = ctx.MkSolver();
foreach (var c in constraints) sGen.Add(c);
var sw = Stopwatch.StartNew();
var rGen = sGen.Check();
sw.Stop();
Console.WriteLine("Solveur generique : " + rGen + " en " + sw.Elapsed.TotalMilliseconds.ToString("F1") + " ms");

var sSpec = ctx.MkSolver("QF_LIA");
foreach (var c in constraints) sSpec.Add(c);
sw = Stopwatch.StartNew();
var rSpec = sSpec.Check();
sw.Stop();
Console.WriteLine("SolverFor QF_LIA  : " + rSpec + " en " + sw.Elapsed.TotalMilliseconds.ToString("F1") + " ms");


Solveur generique : SATISFIABLE en 15,6 ms


SolverFor QF_LIA  : SATISFIABLE en 4,2 ms


## Exercices

Trois exercices à compléter. Les stubs retournent `null` ou `0`.


In [10]:
// EXERCICE 1 : Pipeline de tactiques.
// Appliquer AndThen(simplify, solve-eqs) sur le goal y == x*x ET y > 10.
// Indice : MkGoal, g.Add(...), ctx.AndThen(t1, t2).Apply(g).
// Etape 1 : declarer x, y, goal g
// Etape 2 : g.Add(MkEq(y, MkMul(x, x))) et g.Add(MkGt(y, MkInt(10)))
// Etape 3 : pipeline.Apply(g), retourner Subgoals.Length
int CompterSousGoalsApresPipeline(Context ctx)
{
    // TODO etudiant : implementez le pipeline AndThen(simplify, solve-eqs)
    return 0;  // TODO etudiant : remplacer par le nombre de sous-goals
}

Console.WriteLine("Exercice 1 (pipeline) : " + (CompterSousGoalsApresPipeline(new Context()) > 0 ? CompterSousGoalsApresPipeline(new Context()) + " sous-goal(s)" : "(a completer)"));


Exercice 1 (pipeline) : (a completer)


In [11]:
// EXERCICE 2 : Trouver la clef XOR.
// On a un message chiffre et la clef est un BitVec 8 bits.
// Trouver clef telle que clef XOR message = 0 (clef = message).
// Indice : MkBVXOR(clef, message) == MkBV(0, 8).
// Etape 1 : declarer clef (BitVec 8), fixer message a une valeur connue
// Etape 2 : s.Add(MkEq(MkBVXOR(clef, message), MkBV(0,8)))
// Etape 3 : Check et extraire clef
long? TrouverClefXOR(Context ctx, long message)
{
    // TODO etudiant : implementez la recherche de clef XOR
    return null;  // TODO etudiant : remplacer par la clef trouvee
}

var clef = TrouverClefXOR(new Context(), 42);
Console.WriteLine("Exercice 2 (clef XOR) : " + (clef.HasValue ? clef.ToString() : "(a completer)"));


Exercice 2 (clef XOR) : (a completer)


In [12]:
// EXERCICE 3 : Commutativite des Store sur des index differents.
// Verifier que Store(Store(arr, 0, v0), 1, v1) == Store(Store(arr, 1, v1), 0, v0).
// Indice : MkStore deux fois (index differents), MkEq entre les deux arbres, Check.
// Etape 1 : declarer arr (Array Int->Int), v0, v1
// Etape 2 : construire les deux sequences de Store
// Etape 3 : s.Add(MkEq(seqAB, seqBA)) et Check
string VerifierCommutativiteStore(Context ctx)
{
    // TODO etudiant : implementez la verification de commutativite
    return "(a completer)";  // TODO etudiant : remplacer par "VALIDE" ou "INVALIDE"
}

Console.WriteLine("Exercice 3 (Store commutatif) : " + VerifierCommutativiteStore(new Context()));


Exercice 3 (Store commutatif) : (a completer)


## Conclusion

Ce twin C# couvre les trois familles avancées de Z3 : **tactiques** (`MkTactic`/`MkGoal`/`Apply`/`ApplyResult.Subgoals`, composition `ctx.AndThen`/`OrElse`/`Repeat`), **théorie BitVec** (`MkBVConst`/`MkBVXOR`/`MkBVAND`, arithmétique modulaire, signé vs non signé `MkBVUGt`/`MkBVSgt`, `MkExtract`/`MkConcat` pour la manipulation bit à bit), et **théorie Array** (`MkArrayConst`/`MkStore`/`MkSelect`). Le binding `Microsoft.Z3` expose exactement le même moteur que `z3-solver` Python.

**Complémentarité** : le twin Python utilise `len(result)` et `sg.size()` (API Pythonique) ; ce twin C# montre l'API .NET (`ApplyResult.Subgoals.Length`, `Goal.Formulas` comme propriété, composition via méthodes `ctx.*`) — la **valeur ajoutée** est la traduction des idiomes Z3 dans le système de types .NET.
